In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
INPUT_CSV = "fake_edges_annotated_no_receptors.csv"
OUTPUT_DIR = Path("sampled")

N_REPETITIONS = 5

SELECTED_NODES = ["BCL6", "PU1", "IL9", "RUNX3", "IL22", "GZMB"]

N_EDGES = 5
SEED = 42

In [3]:
df = pd.read_csv(INPUT_CSV)

print(f"Loaded {len(df)} rows")
print("Columns:", list(df.columns))

# Check whether the selected nodes exist at all
existing_nodes = set(df["Regulator"]) | set(df["Target"])

for node in SELECTED_NODES:
    if node in existing_nodes:
        print(f"{node}: found")
    else:
        print(f"{node}: NOT FOUND")

Loaded 7645 rows
Columns: ['Regulator', 'Target', 'Sign', 'Model', 'Confidence Rank', 'Constraint', 'Direct', 'is_kept']
BCL6: found
PU1: found
IL9: found
RUNX3: found
IL22: found
GZMB: found


In [4]:
OUTPUT_DIR.mkdir(exist_ok=True)

# Find candidate rows
selected_mask = (
    df["Regulator"].isin(SELECTED_NODES) &
    df["Target"].isin(SELECTED_NODES)
)

candidates = df[
    selected_mask &
    (df["Confidence Rank"] == 6)
].copy()

if len(candidates) < N_EDGES:
    raise ValueError(
        f"Only {len(candidates)} qualifying edges available, "
        f"but {N_EDGES} requested."
    )

# Generate multiple random selections
for i in range(N_REPETITIONS):
    selected = candidates.sample(
        n=N_EDGES,
        random_state=SEED + i
    ).copy()

    # Change their rank to 5
    selected["Confidence Rank"] = 5

    # Include all rows with confidence < 5
    low_confidence = df[df["Confidence Rank"] < 5]

    output = pd.concat(
        [low_confidence, selected],
        ignore_index=True
    )

    output_file = OUTPUT_DIR / f"selected_nodes_{i+1:02d}.csv"
    output.to_csv(output_file, index=False)

print(f"Created {N_REPETITIONS} files in '{OUTPUT_DIR}'")

Created 5 files in 'sampled'


In [5]:
rank6 = df[df["Confidence Rank"] == 6]

kept = rank6[rank6["is_kept"] == "KEPT"]
taken_out = rank6[rank6["is_kept"] == "TAKEN_OUT"]

n_samples = len(selected)

if len(kept) < n_samples:
    raise ValueError(
        f"Not enough KEPT rank-6 rows: {len(kept)} available, "
        f"but {n_samples} needed."
    )

if len(taken_out) < n_samples:
    raise ValueError(
        f"Not enough TAKEN_OUT rank-6 rows: {len(taken_out)} available, "
        f"but {n_samples} needed."
    )

# These rows are included in every output
original = df[df["Confidence Rank"] < 5].copy()

print(f"Rows sampled per file: {n_samples}")
print(f"KEPT population: {len(kept)}")
print(f"TAKEN_OUT population: {len(taken_out)}")
print(f"Rows copied into every file (confidence < 5): {len(original)}")

Rows sampled per file: 5
KEPT population: 720
TAKEN_OUT population: 6633
Rows copied into every file (confidence < 5): 292


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manual_output = pd.concat(
    [original, selected],
    ignore_index=True
)

manual_output.to_csv(
    OUTPUT_DIR / "selected_nodes.csv",
    index=False
)

In [7]:
for i in range(1, N_REPETITIONS + 1):
    sample = kept.sample(
        n=n_samples,
        replace=False,
        random_state=i
    )

    sample["Confidence Rank"] = 5

    output = pd.concat(
        [original, sample],
        ignore_index=True
    )

    output.to_csv(
        OUTPUT_DIR / f"random_kept_{i:02d}.csv",
        index=False
    )

In [8]:
for i in range(1, N_REPETITIONS + 1):
    sample = taken_out.sample(
        n=n_samples,
        replace=False,
        random_state=i
    )

    sample["Confidence Rank"] = 5

    output = pd.concat(
        [original, sample],
        ignore_index=True
    )

    output.to_csv(
        OUTPUT_DIR / f"random_taken_out_{i:02d}.csv",
        index=False
    )

print(f"Created {N_REPETITIONS} KEPT datasets")
print(f"Created {N_REPETITIONS} TAKEN_OUT datasets")

Created 5 KEPT datasets
Created 5 TAKEN_OUT datasets
